### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

/home/lautaro/ceia/5b2026/pln-i/fiuba-ceia-pln-i/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [26]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  4703, 10870,  4333], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[480.,584.,591.,...,564.,465.,377.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.16,-2.96,-2.95,...,-3. ,-3.19,-3.4 ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 101631)","[[0. ,0.94,0. ,...,0. ,0. ,0. ], [1.39,0.6 ,0. ,...,0. ,0. ,0. ], [0.95,0.14,0. ,...,0. ,0. ,0. ], ..., [0.42,2.9 ,0.04,...,0. ,0. ,0. ], [0.61,1.36,0. ,...,0. ,0. ,0. ], [0.03,0.38,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 101631)","[[-11.56,-10.9 ,-11.56,...,-11.56,-11.56,-11.56], [-10.69,-11.1 ,-11.56,...,-11.56,-11.56,-11.56], [-10.9 ,-11.44,-11.57,...,-11.57,-11.57,-11.57], ..., [-11.22,-10.21,-11.54,...,-11.57,-11.57,-11.57], [-11.09,-10.7 ,-11.56,...,-11.56,-11.56,-11.56], [-11.53,-11.23,-11.56,...,-11.56,-11.56,-11.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,101631


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## Resolución del Desafío 1

### 1. Vectorizar documentos

En general, los documentos que aparecen como similares son de la misma clase, salvo ciertas excepciones debido a que los documentos pueden compartir palabras clave pero no necesariamente pertenecer a la misma categoría temática. Por ejemplo, en doc 4965, que pertenece a la clase 3 (comp.graphics), los documentos más similares pertenecen a la clase 1 (comp.os.ms-windows.misc). Esto se debe a que ambos documentos comparten palabras clave relacionadas con computadoras y tecnología, pero abordan temas diferentes dentro de ese ámbito.

Resulta curioso el caso de doc 7404, donde algunos documentos están formados únicamente por la palabra Hello, lo que genera una alta similaridad entre ellos a pesar de que pertenecen a diferentes clases. Esto demuestra que la similaridad basada en palabras clave puede no reflejar adecuadamente la temática del documento.

In [28]:
# 1. Vectorizar documentos
rng = np.random.default_rng(42)          # semilla fija: resultados reproducibles
docs_idx = rng.choice(X_train.shape[0], size=5, replace=False)

for idx in docs_idx:
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    mostsim = np.argsort(cossim)[::-1][1:6]      # [0] es el propio documento (sim = 1)

    print(f"\n=== doc {idx} [{newsgroups_train.target_names[y_train[idx]]}]")
    print("   ", " ".join(newsgroups_train.data[idx].split())[:160])
    for j in mostsim:
        mismo = y_train[j] == y_train[idx]
        print(f"  {cossim[j]:.3f} [{newsgroups_train.target_names[y_train[j]]}"
              f"{'' if mismo else '  <-- DISTINTA'}]",
              " ".join(newsgroups_train.data[j].split())[:90])



=== doc 8754 [talk.religion.misc]
    /(hudson) /If someone inflicts pain on themselves, whether they enjoy it or not, they /are hurting themselves. They may be permanently damaging their body. That
  0.490 [talk.religion.misc] If I have a habit that I really want to break, and I am willing to make whatever sacrifice
  0.481 [talk.religion.misc] /(hudson) /Yes you do. Who is to say that it is immoral for onesself to experience /pain o
  0.465 [talk.religion.misc] And I maintain: Some people do not want to enter into the light and the knowledge that the
  0.460 [talk.politics.mideast  <-- DISTINTA] [After a small refresh Hasan got on the track again.] |> |> I get the impression Hasan rea
  0.459 [talk.religion.misc] Are you your own master? Do you have any habits that you cannot break? For one, you seem u

=== doc 4965 [comp.sys.mac.hardware]
    No. Plug the printer in the printer port, and the modem in the modem port. ;)
  0.365 [comp.sys.mac.hardware] Battery powered devices like th

### 2. Construir un modelo de clasificación por prototipos (tipo zero-shot)

Se implementó un modelo de clasificación por vecino más cercano (k-NN) con k=1, utilizando la similaridad coseno como métrica de distancia. El modelo asigna a cada documento del conjunto de prueba la clase del documento de entrenamiento más similar.

Algunas limitaciones:
- Alta varianza: con k = 1, un solo vecino ruidoso (un post de una palabra, una cita) decide la predicción.
- Alta dimensionalidad: con 100.000 dimensiones esparsas, las distancias se "aplanan" y a veces el vecino más cercano coincide más por azar que por tema.
- Sin estadística de clase: no resume qué palabras caracterizan cada tema, cosa que sí hace Naïve Bayes. Por eso esperamos que NB gane.

Da un F1-macro de 0.505, bastante por debajo de NB. Acertando el 50% de las veces, pareciera que el modelo está adivinando, pero hay que recordar que essta métrica es macro, por lo que se promedia el F1-score de cada clase:

| Similaridad con el vecino | Documentos | Accuracy
|---------------------------|------------|----------
| 0.0 – 0.2                 | 1128       | 28.5 %
| 0.2 – 0.4                 | 5160       | 49.8 %
| 0.4 – 0.6                 | 918        | 71.6 %
| 0.6 – 1.0                 | 326        | 86.8 %

Cuanto más parecido es el vecino, más confiable la predicción. La media de similaridad es 0.35 en aciertos y 0.26 en errores. El coseno por sí solo no dice si está bien o mal, pero sí ordena la confianza.

Son pares de temas vecinos: el modelo no se equivoca al azar, se confunde entre grupos que hablan de cosas parecidas, como las religiones entre sí o los temas políticos entre sí. Esto es una limitación de la bolsa de palabras: el vocabulario se solapa aunque la postura o el subtema sea distinto.

In [30]:
# 2. Construir un modelo de clasificación por prototipos (tipo zero-shot)
best_idx = np.empty(X_test.shape[0], dtype=int)
best_sim = np.empty(X_test.shape[0])

for start in range(0, X_test.shape[0], 1000):         # por bloques: ahorra memoria
    sims = cosine_similarity(X_test[start:start + 1000], X_train)   # (1000, 11314)
    best_idx[start:start + 1000] = sims.argmax(axis=1)              # el vecino más cercano
    best_sim[start:start + 1000] = sims.max(axis=1)

y_pred_1nn = y_train[best_idx]                        # la clase del vecino
f1_score(y_test, y_pred_1nn, average='macro')


0.5049911553681621

### 3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación

Se compararon `MultinomialNB` y `ComplementNB` variando alpha, sublinear_tf y stop_words (sin modificar ngram_range), eligiendo los hiperparámetros sobre una validación interna del train (80/20 estratificado) y reservando test para la evaluación final. 

Ambos modelos alcanzan prácticamente el mismo F1 macro de validación ≈0.769. En test, MultinomialNB (alpha=0.01, stop_words='english') obtiene F1 macro ≈0.684 y ComplementNB (alpha=0.3, sublinear_tf=True) ≈0.699, frente a 0.585 del modelo base. 

La mejora proviene sobre todo del ajuste de alpha (con TF-IDF, α=1 suaviza en exceso) y del uso de ComplementNB, que además es menos sensible a ese hiperparámetro. 

`ComplementNB` destaca por encima de `MultinomialNB` al ser más estable ante clases desbalanceadas, ya que hace el foco en las demás clases (complemento) y no utiliza el prior de clase como lo hace MultinomialNB.

In [37]:
from sklearn.model_selection import train_test_split

# 1) Validación interna: elegimos hiperparámetros sin mirar test
txt_tr, txt_val, y_tr, y_val = train_test_split(
    newsgroups_train.data, y_train, test_size=0.2, stratify=y_train, random_state=0)

resultados = []
for sublinear in [False, True]:
    for stop in [None, 'english']:
        vec = TfidfVectorizer(sublinear_tf=sublinear, stop_words=stop)  # ngram_range sin modificar
        Xa, Xv = vec.fit_transform(txt_tr), vec.transform(txt_val)
        for nombre, Modelo in [('Multinomial', MultinomialNB), ('Complement', ComplementNB)]:
            for alpha in [0.01, 0.03, 0.1, 0.3, 1.0]:
                f1 = f1_score(y_val, Modelo(alpha=alpha).fit(Xa, y_tr).predict(Xv), average='macro')
                resultados.append((f1, nombre, alpha, sublinear, stop))

resultados.sort(reverse=True)
for r in resultados[:5]: print(r)


(0.7687038788586211, 'Multinomial', 0.01, False, 'english')
(0.7684728634558081, 'Complement', 0.3, True, None)
(0.7681894968743119, 'Multinomial', 0.01, False, None)
(0.7662724516071701, 'Complement', 0.3, False, None)
(0.7645173722179573, 'Multinomial', 0.01, True, None)


In [38]:
# 2) Comparar en test los dos mejores
for nombre, Modelo, alpha, sub, stop in [
    ('Multinomial', MultinomialNB, 0.01, False, 'english'),
    ('Complement',  ComplementNB,  0.3,  True,  None)]:
    vec = TfidfVectorizer(sublinear_tf=sub, stop_words=stop)
    clf = Modelo(alpha=alpha).fit(vec.fit_transform(newsgroups_train.data), y_train)
    print(nombre, f1_score(y_test, clf.predict(vec.transform(newsgroups_test.data)), average='macro'))


Multinomial 0.6844389919212164
Complement 0.6989825595098211


In [39]:
# 3) Reentrenar con train y reportar test una sola vez
vec = TfidfVectorizer(sublinear_tf=True)
clf = ComplementNB(alpha=0.3).fit(vec.fit_transform(newsgroups_train.data), y_train)
f1_score(y_test, clf.predict(vec.transform(newsgroups_test.data)), average='macro')

0.6989825595098211

### 4. Transponer la matriz documento-término

Se traspone la matriz documento-término (`X_train.T`): ahora cada fila es una palabra y cada columna un documento, así que una palabra queda representada por los documentos en los que aparece. Para cinco palabras elegidas manualmente (`car`, `god`, `space`, `windows`, `israel`) calculamos la similaridad coseno contra todas las demás y listamos las cinco más cercanas, junto con en cuántos documentos aparece cada una.

**Observaciones:**
- Los vecinos suelen formar campos temáticos coherentes: `israel` → israeli, arab, syrian, lebanon; `space` → nasa, shuttle, seti; `god` → jesus, bible, christ. Dos palabras se parecen cuando aparecen en los mismos documentos (hipótesis distribucional).
- La similaridad es temática y no de significado: `car` se acerca a `dealer` u `owner`, no a sinónimos.
- Aparece ruido. Palabras muy raras (`criterium` en `car`, `seds` y `enfant` en `space`, presentes en 1 a 4 documentos) coinciden por azar en un solo texto. Palabras muy frecuentes (`that` en `god`, `for` en `windows`) coinciden con casi cualquier palabra.
- `windows` solo devuelve el sentido de sistema operativo (dos, microsoft, nt), porque el corpus es de foros de informática y porque un único vector por palabra no distingue sentidos.

**Conclusión:** La co-ocurrencia en documentos captura relaciones temáticas, pero con vectores muy grandes y esparsos, sensibles a términos raros y frecuentes, y con un solo vector por palabra. Esto motiva los embeddings densos (Word2Vec, GloVe) que se verán más adelante.


In [40]:
X_train_T = X_train.T.tocsr()          # términos x documentos (.T da CSC; CSR permite sacar filas rápido)
df = np.diff(X_train_T.indptr)         # nº de elementos no nulos por fila = en cuántos documentos aparece cada término

def vecinos(palabra, k=5):
    i = tfidfvect.vocabulary_[palabra]
    sims = cosine_similarity(X_train_T[i], X_train_T)[0]     # (101631,): la palabra contra todas
    top = np.argsort(sims)[::-1][1:k + 1]                    # [0] es la propia palabra
    return [(idx2word[j], round(sims[j], 2), df[j]) for j in top]

for w in ['car', 'god', 'space', 'windows', 'israel']:       # elegidas a mano
    print(f"{w} (en {df[tfidfvect.vocabulary_[w]]} docs):", vecinos(w))


car (en 396 docs): [('cars', np.float64(0.18), np.int32(177)), ('criterium', np.float64(0.18), np.int32(1)), ('civic', np.float64(0.17), np.int32(22)), ('owner', np.float64(0.17), np.int32(99)), ('dealer', np.float64(0.17), np.int32(107))]
god (en 579 docs): [('jesus', np.float64(0.27), np.int32(263)), ('bible', np.float64(0.26), np.int32(243)), ('that', np.float64(0.26), np.int32(6580)), ('existence', np.float64(0.25), np.int32(137)), ('christ', np.float64(0.25), np.int32(197))]
space (en 398 docs): [('nasa', np.float64(0.33), np.int32(157)), ('seds', np.float64(0.3), np.int32(3)), ('shuttle', np.float64(0.29), np.int32(71)), ('enfant', np.float64(0.28), np.int32(4)), ('seti', np.float64(0.25), np.int32(4))]
windows (en 575 docs): [('dos', np.float64(0.3), np.int32(281)), ('ms', np.float64(0.23), np.int32(177)), ('microsoft', np.float64(0.22), np.int32(98)), ('nt', np.float64(0.21), np.int32(79)), ('for', np.float64(0.19), np.int32(6294))]
israel (en 184 docs): [('israeli', np.float64